In [3]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("challenges").getOrCreate()
print("Spark is created")

Spark is created


In [4]:
from google.colab import files
uploaded=files.upload()

Saving employees.csv to employees.csv
Saving logins.txt to logins.txt
Saving orders.json to orders.json
Saving sales.csv to sales.csv


In [5]:
log_df=spark.read.text("logins.txt")
log_df.show()

+-----+
|value|
+-----+
|Rahul|
|Sneha|
|Arjun|
|Rahul|
|Priya|
|Sneha|
|Rahul|
|Karan|
|Arjun|
|Sneha|
|Rahul|
| Amit|
|Priya|
|Karan|
|Sneha|
|Rahul|
|Meera|
|Arjun|
|Sneha|
|Rahul|
+-----+
only showing top 20 rows


In [8]:
print("Total logins",log_df.count())

Total logins 26


In [9]:
log_df.distinct().show()

+-----+
|value|
+-----+
|Meera|
|Sneha|
|Priya|
|Rahul|
|Arjun|
| Amit|
|Karan|
+-----+



In [12]:
from pyspark.sql.functions import col
log_count=log_df.groupBy("value").count()
log_count.show()

+-----+-----+
|value|count|
+-----+-----+
|Meera|    1|
|Sneha|    6|
|Priya|    3|
|Rahul|    7|
|Arjun|    4|
| Amit|    2|
|Karan|    3|
+-----+-----+



In [13]:
log_count.orderBy(col("count").desc()).show(3)

+-----+-----+
|value|count|
+-----+-----+
|Rahul|    7|
|Sneha|    6|
|Arjun|    4|
+-----+-----+
only showing top 3 rows


In [14]:
log_count.filter(col("count")>4).show()

+-----+-----+
|value|count|
+-----+-----+
|Sneha|    6|
|Rahul|    7|
+-----+-----+



In [15]:
log_dict = {row["value"]: row["count"] for row in log_count.collect()}
print(log_dict)

{'Meera': 1, 'Sneha': 6, 'Priya': 3, 'Rahul': 7, 'Arjun': 4, 'Amit': 2, 'Karan': 3}


In [16]:
emp_df=spark.read.csv("employees.csv",header=True,inferSchema=True)
emp_df.show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     1| Rahul|        IT| 70000|Hyderabad|
|     2| Sneha|        HR| 60000|Bangalore|
|     3| Arjun|        IT| 75000|  Chennai|
|     4| Priya|   Finance| 80000|Hyderabad|
|     5| Karan|        IT| 50000|   Mumbai|
|     6|  Amit|        HR| 58000|    Delhi|
|     7| Meera|   Finance| 82000|Bangalore|
|     8|  Ravi|        IT| 72000|Hyderabad|
|     9|  Neha|        HR| 61000|  Chennai|
|    10|Vikram|   Finance| 90000|    Delhi|
|    11| Anita|        IT| 65000|Bangalore|
|    12| Manoj|        HR| 62000|   Mumbai|
|    13| Divya|        IT| 77000|Hyderabad|
|    14|Sanjay|   Finance| 85000|  Chennai|
|    15| Pooja|        IT| 69000|Bangalore|
|    16| Kunal|        HR| 61000|    Delhi|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    18|Deepak|        IT| 73000|Hyderabad|
|    19|  Ritu|        HR| 60000|  Chennai|
|    20| Akash|   Finance| 91000

In [17]:
emp_df.count()

20

In [24]:
emp_df.filter(emp_df.department=="IT").show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     1| Rahul|        IT| 70000|Hyderabad|
|     3| Arjun|        IT| 75000|  Chennai|
|     5| Karan|        IT| 50000|   Mumbai|
|     8|  Ravi|        IT| 72000|Hyderabad|
|    11| Anita|        IT| 65000|Bangalore|
|    13| Divya|        IT| 77000|Hyderabad|
|    15| Pooja|        IT| 69000|Bangalore|
|    18|Deepak|        IT| 73000|Hyderabad|
+------+------+----------+------+---------+



In [25]:
emp_df.filter(emp_df.salary>75000).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     4| Priya|   Finance| 80000|Hyderabad|
|     7| Meera|   Finance| 82000|Bangalore|
|    10|Vikram|   Finance| 90000|    Delhi|
|    13| Divya|        IT| 77000|Hyderabad|
|    14|Sanjay|   Finance| 85000|  Chennai|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    20| Akash|   Finance| 91000|    Delhi|
+------+------+----------+------+---------+



In [31]:
from pyspark.sql.functions import avg,max,min,col
emp_df.select(avg("salary")).show()
emp_df.orderBy(col("salary").desc()).limit(1).show()
emp_df.select(min("salary")).show()

+-----------+
|avg(salary)|
+-----------+
|    71450.0|
+-----------+

+------+-----+----------+------+-----+
|emp_id| name|department|salary| city|
+------+-----+----------+------+-----+
|    20|Akash|   Finance| 91000|Delhi|
+------+-----+----------+------+-----+

+-----------+
|min(salary)|
+-----------+
|      50000|
+-----------+



In [33]:
emp_df.groupBy("department").count().show()

+----------+-----+
|department|count|
+----------+-----+
|        HR|    6|
|   Finance|    6|
|        IT|    8|
+----------+-----+



In [34]:
from pyspark.sql.functions import avg
emp_df.groupBy("department").avg("salary").show()

+----------+------------------+
|department|       avg(salary)|
+----------+------------------+
|        HR|60333.333333333336|
|   Finance|           86000.0|
|        IT|           68875.0|
+----------+------------------+



In [35]:
emp_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    4|
|  Chennai|    4|
|   Mumbai|    3|
|    Delhi|    4|
|Hyderabad|    5|
+---------+-----+



In [37]:
emp_df.orderBy(col("salary").desc()).limit(5).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|    20| Akash|   Finance| 91000|    Delhi|
|    10|Vikram|   Finance| 90000|    Delhi|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    14|Sanjay|   Finance| 85000|  Chennai|
|     7| Meera|   Finance| 82000|Bangalore|
+------+------+----------+------+---------+



In [38]:
emp_df.filter((emp_df.city=="Hyderabad")&(emp_df.salary>70000)).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     4| Priya|   Finance| 80000|Hyderabad|
|     8|  Ravi|        IT| 72000|Hyderabad|
|    13| Divya|        IT| 77000|Hyderabad|
|    18|Deepak|        IT| 73000|Hyderabad|
+------+------+----------+------+---------+



In [41]:
sales_df=spark.read.csv("sales.csv",header=True,inferSchema=True)
sales_df.show()

+-------+------+--------+--------+-----+
|sale_id|emp_id| product|quantity|price|
+-------+------+--------+--------+-----+
|      1|     1|  Laptop|       1|75000|
|      2|     2|   Mouse|       3|  500|
|      3|     3|Keyboard|       2| 1500|
|      4|     1| Monitor|       1|12000|
|      5|     4|  Laptop|       1|75000|
|      6|     3|   Mouse|       2|  500|
|      7|     5|Keyboard|       1| 1500|
|      8|     1|  Laptop|       1|75000|
|      9|     6|   Mouse|       4|  500|
|     10|     7| Monitor|       2|12000|
|     11|     8|Keyboard|       2| 1500|
|     12|     9|   Mouse|       3|  500|
|     13|    10|  Laptop|       1|75000|
|     14|    11| Monitor|       1|12000|
|     15|    12|Keyboard|       2| 1500|
|     16|    13|  Laptop|       1|75000|
|     17|    14|   Mouse|       3|  500|
|     18|    15| Monitor|       1|12000|
|     19|    16|Keyboard|       1| 1500|
|     20|    17|  Laptop|       1|75000|
+-------+------+--------+--------+-----+



In [53]:
sales_df=sales_df.withColumn("revenue",col("quantity")*col("price"))
sales_df.show()

+-------+------+--------+--------+-----+-------+
|sale_id|emp_id| product|quantity|price|revenue|
+-------+------+--------+--------+-----+-------+
|      1|     1|  Laptop|       1|75000|  75000|
|      2|     2|   Mouse|       3|  500|   1500|
|      3|     3|Keyboard|       2| 1500|   3000|
|      4|     1| Monitor|       1|12000|  12000|
|      5|     4|  Laptop|       1|75000|  75000|
|      6|     3|   Mouse|       2|  500|   1000|
|      7|     5|Keyboard|       1| 1500|   1500|
|      8|     1|  Laptop|       1|75000|  75000|
|      9|     6|   Mouse|       4|  500|   2000|
|     10|     7| Monitor|       2|12000|  24000|
|     11|     8|Keyboard|       2| 1500|   3000|
|     12|     9|   Mouse|       3|  500|   1500|
|     13|    10|  Laptop|       1|75000|  75000|
|     14|    11| Monitor|       1|12000|  12000|
|     15|    12|Keyboard|       2| 1500|   3000|
|     16|    13|  Laptop|       1|75000|  75000|
|     17|    14|   Mouse|       3|  500|   1500|
|     18|    15| Mon

In [48]:
sales_df.selectExpr("sum(quantity * price) as total_revenue").show()

+-------------+
|total_revenue|
+-------------+
|       529500|
+-------------+



In [50]:
sales_df.groupBy("product").sum("quantity").show()

+--------+-------------+
| product|sum(quantity)|
+--------+-------------+
|  Laptop|            6|
|   Mouse|           15|
|Keyboard|            8|
| Monitor|            5|
+--------+-------------+



In [51]:
sales_df.groupBy("product").sum("quantity") \
    .orderBy(col("sum(quantity)").desc()).show(1)

+-------+-------------+
|product|sum(quantity)|
+-------+-------------+
|  Mouse|           15|
+-------+-------------+
only showing top 1 row


In [54]:
sales_df.groupBy("emp_id").sum("revenue") \
    .orderBy(col("sum(revenue)").desc()).show(1)

+------+------------+
|emp_id|sum(revenue)|
+------+------------+
|     1|      162000|
+------+------------+
only showing top 1 row


In [55]:
sales_df.select(avg("revenue")).show()

+------------+
|avg(revenue)|
+------------+
|     26475.0|
+------------+



In [59]:
product_revenue = sales_df.groupBy("product").sum("revenue")
product_revenue.show()

+--------+------------+
| product|sum(revenue)|
+--------+------------+
|  Laptop|      450000|
|   Mouse|        7500|
|Keyboard|       12000|
| Monitor|       60000|
+--------+------------+



In [61]:
product_revenue.filter(col("sum(revenue)") > 100000).show()

+-------+------------+
|product|sum(revenue)|
+-------+------------+
| Laptop|      450000|
+-------+------------+



In [82]:
from google.colab import files
uploaded=files.upload()

Saving or.json to or.json


In [95]:
from pyspark.sql.functions import explode

orders_df_raw = spark.read.option("multiLine", "true").json("or.json")

df = orders_df_raw

print("Schema of the correctly parsed orders DataFrame:")
df.printSchema()

print("First 10 rows of the correctly parsed orders DataFrame:")
df.show()

Schema of the correctly parsed orders DataFrame:
root
 |-- amount: long (nullable = true)
 |-- city: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- product: string (nullable = true)

First 10 rows of the correctly parsed orders DataFrame:
+------+---------+--------+--------+--------+
|amount|     city|customer|order_id| product|
+------+---------+--------+--------+--------+
| 75000|Hyderabad|   Rahul|       1|  Laptop|
|  1500|Bangalore|   Sneha|       2|   Mouse|
|  3000|  Chennai|   Arjun|       3|Keyboard|
| 75000|Hyderabad|   Priya|       4|  Laptop|
| 12000|   Mumbai|   Karan|       5| Monitor|
|  1000|Hyderabad|   Rahul|       6|   Mouse|
| 75000|Bangalore|   Sneha|       7|  Laptop|
|  3000|  Chennai|   Arjun|       8|Keyboard|
|  2000|Hyderabad|   Priya|       9|   Mouse|
| 12000|Hyderabad|   Rahul|      10| Monitor|
+------+---------+--------+--------+--------+



In [96]:
df.count()

10

In [99]:
df.select(sum("amount")).show()

+-----------+
|sum(amount)|
+-----------+
|     259500|
+-----------+



In [101]:
df.groupBy("customer").sum("amount").show()

+--------+-----------+
|customer|sum(amount)|
+--------+-----------+
|   Sneha|      76500|
|   Priya|      77000|
|   Rahul|      88000|
|   Arjun|       6000|
|   Karan|      12000|
+--------+-----------+



In [102]:
df.groupBy("customer").sum("amount") \
    .orderBy(col("sum(amount)").desc()).show(1)

+--------+-----------+
|customer|sum(amount)|
+--------+-----------+
|   Rahul|      88000|
+--------+-----------+
only showing top 1 row


In [103]:
df.groupBy("product").sum("amount").show()

+--------+-----------+
| product|sum(amount)|
+--------+-----------+
|  Laptop|     225000|
|   Mouse|       4500|
|Keyboard|       6000|
| Monitor|      24000|
+--------+-----------+



In [104]:
df.filter(df.city=="Hyderabad").show()

+------+---------+--------+--------+-------+
|amount|     city|customer|order_id|product|
+------+---------+--------+--------+-------+
| 75000|Hyderabad|   Rahul|       1| Laptop|
| 75000|Hyderabad|   Priya|       4| Laptop|
|  1000|Hyderabad|   Rahul|       6|  Mouse|
|  2000|Hyderabad|   Priya|       9|  Mouse|
| 12000|Hyderabad|   Rahul|      10|Monitor|
+------+---------+--------+--------+-------+



In [105]:
df.filter(df.amount>10000).show()

+------+---------+--------+--------+-------+
|amount|     city|customer|order_id|product|
+------+---------+--------+--------+-------+
| 75000|Hyderabad|   Rahul|       1| Laptop|
| 75000|Hyderabad|   Priya|       4| Laptop|
| 12000|   Mumbai|   Karan|       5|Monitor|
| 75000|Bangalore|   Sneha|       7| Laptop|
| 12000|Hyderabad|   Rahul|      10|Monitor|
+------+---------+--------+--------+-------+



In [106]:
df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    2|
|   Mumbai|    1|
|Hyderabad|    5|
+---------+-----+



In [108]:
from posixpath import join
joined_df = emp_df.join(sales_df, on="emp_id")
joined_df.show()

+------+------+----------+------+---------+-------+--------+--------+-----+-------+
|emp_id|  name|department|salary|     city|sale_id| product|quantity|price|revenue|
+------+------+----------+------+---------+-------+--------+--------+-----+-------+
|     1| Rahul|        IT| 70000|Hyderabad|      8|  Laptop|       1|75000|  75000|
|     1| Rahul|        IT| 70000|Hyderabad|      4| Monitor|       1|12000|  12000|
|     1| Rahul|        IT| 70000|Hyderabad|      1|  Laptop|       1|75000|  75000|
|     2| Sneha|        HR| 60000|Bangalore|      2|   Mouse|       3|  500|   1500|
|     3| Arjun|        IT| 75000|  Chennai|      6|   Mouse|       2|  500|   1000|
|     3| Arjun|        IT| 75000|  Chennai|      3|Keyboard|       2| 1500|   3000|
|     4| Priya|   Finance| 80000|Hyderabad|      5|  Laptop|       1|75000|  75000|
|     5| Karan|        IT| 50000|   Mumbai|      7|Keyboard|       1| 1500|   1500|
|     6|  Amit|        HR| 58000|    Delhi|      9|   Mouse|       4|  500| 

In [110]:
emp_revenue = joined_df.withColumn("revenue", col("quantity") * col("price")) \
    .groupBy("name").sum("revenue")

emp_revenue.show()

+------+------------+
|  name|sum(revenue)|
+------+------------+
| Kunal|        1500|
| Sonal|       75000|
| Divya|       75000|
|  Ravi|        3000|
|Sanjay|        1500|
| Meera|       24000|
| Sneha|        1500|
| Priya|       75000|
|Vikram|       75000|
| Rahul|      162000|
| Anita|       12000|
| Manoj|        3000|
| Pooja|       12000|
| Arjun|        4000|
|  Amit|        2000|
|  Neha|        1500|
| Karan|        1500|
+------+------------+



In [111]:
emp_revenue.orderBy(col("sum(revenue)").desc()).show(5)

+------+------------+
|  name|sum(revenue)|
+------+------------+
| Rahul|      162000|
| Priya|       75000|
| Divya|       75000|
| Sonal|       75000|
|Vikram|       75000|
+------+------------+
only showing top 5 rows


In [112]:
dept_revenue = joined_df.withColumn("revenue", col("quantity") * col("price")) \
    .groupBy("department").sum("revenue") \
    .orderBy(col("sum(revenue)").desc())

dept_revenue.show(1)

+----------+------------+
|department|sum(revenue)|
+----------+------------+
|        IT|      269500|
+----------+------------+
only showing top 1 row


In [113]:
emp_revenue.write.csv("final_sales_report.csv", header=True)